In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from mal_client import MALClient
from anime_data import AnimeDataClient
from anime_recommender import SimilarityRecommender

load_dotenv(PROJECT_ROOT / ".env")

client_id = os.getenv("CLIENT_ID")

In [3]:
anime_data_client = AnimeDataClient(client_id, cache_file=PROJECT_ROOT / "anime_cache.json")

In [4]:
anime_data = anime_data_client.get_cache()

Build features

In [5]:
from anime_features import AnimeFeatureBuilder

builder = AnimeFeatureBuilder(
    anime_data,
    max_tfidf_features=3000,
    n_svd_components=300
)

anime_df = builder.build_features()

builder.svd_explained_variance

np.float64(0.37937105892729933)

Convert each anime in df to vectors

In [6]:
recommender = SimilarityRecommender()
anime_vectors = recommender.create_anime_vectors(anime_df)
anime_df_scaled = recommender.anime_df_scaled

Get user Data

In [7]:
username = "chekkit"
user_client = MALClient(client_id)

user_data = user_client.get_user_data(username)
user_scores = user_client.get_scores(user_data)

Tuning Bayesian

In [8]:
weights_uncertainty = np.array([
    0, 1, 2, 3,
    3.5, 4, 4.5, 5, 5.5, 6, 6.5, 7, 7.5, 8,
    8.5, 9, 10, 11, 12, 13, 14, 15, 16
])
n_runs = 1000
tuning_top_ks = [5, 10]

from anime_evaluation import HitRateEvaluator

hitman = HitRateEvaluator(
            anime_df_scaled=anime_df_scaled,
            anime_df=anime_df,
            scores=user_scores,
            anime_data_client=anime_data_client,
            anime_data=anime_data,
            builder=builder,
            recommender=recommender,
        )

(
    bayesian_results,
    bayesian_summary,
    best_bayesian_weights,
    baseline_results,
    baseline_summary,
) = hitman.tune_bayesian_uncertainty(
    weights=weights_uncertainty,
    n_runs=n_runs,
    top_ks=tuning_top_ks,
    random_state=42,
)

best_bayesian_weights = best_bayesian_weights.rename(
    columns={"uncertainty_weight": "bayesian_uncertainty_weight"}
)

average_metrics = bayesian_summary.merge(
    baseline_summary,
    on="k",
    how="left",
).rename(columns={"uncertainty_weight": "bayesian_uncertainty_weight"})

best_bayesian_weights

,bayesian_uncertainty_weight,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits,baseline_avg_precision_at_k,baseline_std_precision_at_k,baseline_avg_hit_rate,baseline_std_hit_rate,baseline_avg_hits
0,15.0,5,0.5774,0.201121,0.090219,0.031425,2.887,0.1024,0.124297,0.016,0.019421,0.512
1,16.0,10,0.4097,0.143026,0.128031,0.044696,4.097,0.0512,0.062148,0.016,0.019421,0.512


In [9]:
bayesian_summary

,uncertainty_weight,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits
42,15.0,5,0.5774,0.201121,0.090219,0.031425,2.887
44,16.0,5,0.5768,0.198548,0.090125,0.031023,2.884
40,14.0,5,0.5732,0.203871,0.089563,0.031855,2.866
38,13.0,5,0.5726,0.205649,0.089469,0.032133,2.863
36,12.0,5,0.5712,0.211885,0.089250,0.033107,2.856
34,11.0,5,0.5650,0.213684,0.088281,0.033388,2.825
32,10.0,5,0.5510,0.211858,0.086094,0.033103,2.755
30,9.0,5,0.5192,0.206867,0.081125,0.032323,2.596
28,8.5,5,0.5040,0.201853,0.078750,0.031540,2.520
26,8.0,5,0.4862,0.192267,0.075969,0.030042,2.431


In [ ]:
focused_weights_uncertainty = np.array([15, 16, 17, 18, 19, 20, 21])
focused_n_runs = 500
focused_top_ks = [5]

from anime_evaluation import HitRateEvaluator, RankingMetricEvaluator

focused_hitman = HitRateEvaluator(
    anime_df_scaled=anime_df_scaled,
    anime_df=anime_df,
    scores=user_scores,
    anime_data_client=anime_data_client,
    anime_data=anime_data,
    builder=builder,
    recommender=recommender,
)

(
    focused_bayesian_results,
    focused_bayesian_summary,
    focused_best_bayesian_weights,
    focused_baseline_results,
    focused_baseline_summary,
) = focused_hitman.tune_bayesian_uncertainty(
    weights=focused_weights_uncertainty,
    n_runs=focused_n_runs,
    top_ks=focused_top_ks,
    random_state=42,
)

focused_ranking_evaluator = RankingMetricEvaluator(
    anime_df_scaled=anime_df_scaled,
    anime_df=anime_df,
    scores=user_scores,
    anime_data_client=anime_data_client,
    anime_data=anime_data,
    builder=builder,
    recommender=recommender,
)

focused_ranking_results, focused_ranking_summary = (
    focused_ranking_evaluator.tune_bayesian_uncertainty_ranking(
        weights=focused_weights_uncertainty,
        n_runs=focused_n_runs,
        top_ks=focused_top_ks,
        random_state=42,
    )
)

focused_bayesian_summary = focused_bayesian_summary.rename(
    columns={"uncertainty_weight": "bayesian_uncertainty_weight"}
)
focused_ranking_summary = focused_ranking_summary.rename(
    columns={"uncertainty_weight": "bayesian_uncertainty_weight"}
)

focused_average_metrics = (
    focused_bayesian_summary
    .merge(
        focused_ranking_summary,
        on=["bayesian_uncertainty_weight", "k"],
        how="left",
    )
    .merge(
        focused_baseline_summary,
        on="k",
        how="left",
    )
)

focused_metrics_path = (
    PROJECT_ROOT
    / "metrics"
    / "current_corpus_4793_anime"
    / f"bayesian_uncertainty_focused_ndcg_{focused_n_runs}run_202606.csv"
)
focused_average_metrics.to_csv(focused_metrics_path, index=False)

focused_best_bayesian_weights = (
    focused_average_metrics
    .sort_values(
        ["k", "avg_precision_at_k", "avg_ndcg_at_k", "avg_hit_rate"],
        ascending=[True, False, False, False],
    )
    .groupby("k")
    .head(1)
)

focused_best_bayesian_weights

,bayesian_uncertainty_weight,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits,avg_ndcg_at_k,std_ndcg_at_k,avg_mrr_at_k,std_mrr_at_k,avg_relevant_hits_at_k,avg_strong_hits_at_k,avg_test_relevant,avg_test_strong_relevant,baseline_avg_precision_at_k,baseline_std_precision_at_k,baseline_avg_hit_rate,baseline_std_hit_rate,baseline_avg_hits
0,20,5,0.5824,0.201025,0.091,0.03141,2.912,0.655495,0.19666,0.973233,0.131024,3.414,2.66,46.326,31.87,0.0976,0.121093,0.01525,0.018921,0.488


In [13]:
focused_average_metrics

,bayesian_uncertainty_weight,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits,avg_ndcg_at_k,std_ndcg_at_k,avg_mrr_at_k,std_mrr_at_k,avg_relevant_hits_at_k,avg_strong_hits_at_k,avg_test_relevant,avg_test_strong_relevant,baseline_avg_precision_at_k,baseline_std_precision_at_k,baseline_avg_hit_rate,baseline_std_hit_rate,baseline_avg_hits
0,20,5,0.5824,0.201025,0.091000,0.031410,2.912,0.655495,0.196660,0.973233,0.131024,3.414,2.660,46.326,31.87,0.0976,0.121093,0.01525,0.018921,0.488
1,21,5,0.5820,0.199589,0.090938,0.031186,2.910,0.654644,0.198153,0.974567,0.127853,3.414,2.660,46.326,31.87,0.0976,0.121093,0.01525,0.018921,0.488
2,19,5,0.5784,0.201829,0.090375,0.031536,2.892,0.654388,0.197059,0.972900,0.132435,3.402,2.648,46.326,31.87,0.0976,0.121093,0.01525,0.018921,0.488
3,18,5,0.5768,0.205197,0.090125,0.032062,2.884,0.655659,0.197208,0.973300,0.130087,3.400,2.664,46.326,31.87,0.0976,0.121093,0.01525,0.018921,0.488
4,15,5,0.5728,0.203132,0.089500,0.031739,2.864,0.657494,0.197168,0.969567,0.137193,3.348,2.698,46.326,31.87,0.0976,0.121093,0.01525,0.018921,0.488
5,16,5,0.5716,0.199180,0.089313,0.031122,2.858,0.657998,0.195821,0.972900,0.129888,3.376,2.690,46.326,31.87,0.0976,0.121093,0.01525,0.018921,0.488
6,17,5,0.5716,0.202374,0.089313,0.031621,2.858,0.657238,0.196946,0.972633,0.130055,3.392,2.686,46.326,31.87,0.0976,0.121093,0.01525,0.018921,0.488


In [14]:
focused_weights_uncertainty = np.array([22, 23, 24, 25])
focused_n_runs = 500
focused_top_ks = [5]

from anime_evaluation import HitRateEvaluator, RankingMetricEvaluator

focused_hitman = HitRateEvaluator(
    anime_df_scaled=anime_df_scaled,
    anime_df=anime_df,
    scores=user_scores,
    anime_data_client=anime_data_client,
    anime_data=anime_data,
    builder=builder,
    recommender=recommender,
)

(
    focused_bayesian_results,
    focused_bayesian_summary,
    focused_best_bayesian_weights,
    focused_baseline_results,
    focused_baseline_summary,
) = focused_hitman.tune_bayesian_uncertainty(
    weights=focused_weights_uncertainty,
    n_runs=focused_n_runs,
    top_ks=focused_top_ks,
    random_state=42,
)

focused_ranking_evaluator = RankingMetricEvaluator(
    anime_df_scaled=anime_df_scaled,
    anime_df=anime_df,
    scores=user_scores,
    anime_data_client=anime_data_client,
    anime_data=anime_data,
    builder=builder,
    recommender=recommender,
)

focused_ranking_results, focused_ranking_summary = (
    focused_ranking_evaluator.tune_bayesian_uncertainty_ranking(
        weights=focused_weights_uncertainty,
        n_runs=focused_n_runs,
        top_ks=focused_top_ks,
        random_state=42,
    )
)

focused_bayesian_summary = focused_bayesian_summary.rename(
    columns={"uncertainty_weight": "bayesian_uncertainty_weight"}
)
focused_ranking_summary = focused_ranking_summary.rename(
    columns={"uncertainty_weight": "bayesian_uncertainty_weight"}
)

focused_average_metrics = (
    focused_bayesian_summary
    .merge(
        focused_ranking_summary,
        on=["bayesian_uncertainty_weight", "k"],
        how="left",
    )
    .merge(
        focused_baseline_summary,
        on="k",
        how="left",
    )
)

focused_metrics_path = (
    PROJECT_ROOT
    / "metrics"
    / "current_corpus_4793_anime"
    / f"bayesian_uncertainty_focused_ndcg_{focused_n_runs}run2_202606.csv"
)
focused_average_metrics.to_csv(focused_metrics_path, index=False)

focused_best_bayesian_weights = (
    focused_average_metrics
    .sort_values(
        ["k", "avg_precision_at_k", "avg_ndcg_at_k", "avg_hit_rate"],
        ascending=[True, False, False, False],
    )
    .groupby("k")
    .head(1)
)

focused_best_bayesian_weights

,bayesian_uncertainty_weight,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits,avg_ndcg_at_k,std_ndcg_at_k,avg_mrr_at_k,std_mrr_at_k,avg_relevant_hits_at_k,avg_strong_hits_at_k,avg_test_relevant,avg_test_strong_relevant,baseline_avg_precision_at_k,baseline_std_precision_at_k,baseline_avg_hit_rate,baseline_std_hit_rate,baseline_avg_hits
0,23,5,0.5812,0.195865,0.090813,0.030604,2.906,0.650508,0.196812,0.972567,0.128092,3.41,2.632,46.326,31.87,0.0976,0.121093,0.01525,0.018921,0.488


In [15]:
focused_average_metrics

,bayesian_uncertainty_weight,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits,avg_ndcg_at_k,std_ndcg_at_k,avg_mrr_at_k,std_mrr_at_k,avg_relevant_hits_at_k,avg_strong_hits_at_k,avg_test_relevant,avg_test_strong_relevant,baseline_avg_precision_at_k,baseline_std_precision_at_k,baseline_avg_hit_rate,baseline_std_hit_rate,baseline_avg_hits
0,23,5,0.5812,0.195865,0.090813,0.030604,2.906,0.650508,0.196812,0.972567,0.128092,3.410,2.632,46.326,31.87,0.0976,0.121093,0.01525,0.018921,0.488
1,22,5,0.5800,0.196360,0.090625,0.030681,2.900,0.652722,0.197051,0.974567,0.124545,3.416,2.644,46.326,31.87,0.0976,0.121093,0.01525,0.018921,0.488
2,24,5,0.5792,0.195047,0.090500,0.030476,2.896,0.649503,0.196757,0.972400,0.128652,3.408,2.624,46.326,31.87,0.0976,0.121093,0.01525,0.018921,0.488
3,25,5,0.5760,0.194267,0.090000,0.030354,2.880,0.647455,0.195534,0.973900,0.124853,3.400,2.606,46.326,31.87,0.0976,0.121093,0.01525,0.018921,0.488


## Results

This current-corpus tuning used the selected feature set **mean + popularity + watching + genres + synopsis SVD** with Bayesian Ridge, `clip_predictions=False`, and the current 4,793-anime corpus. The broad sweep first tested weights through `16`, then two focused 500-run sweeps checked the high-weight region from `15` to `25` with Precision@5 plus NDCG/MRR.

| Run | Weights tested | Best weight | P@5 | Hits@5 | NDCG@5 | MRR@5 | Baseline P@5 | Notes |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | --- |
| Broad sweep | `0..16` | 15 | 0.5774 | 2.887 | n/a | n/a | 0.1024 | Showed the useful region moved much higher on the current corpus. |
| Focused sweep 1 | `15..21` | 20 | 0.5824 | 2.912 | 0.6555 | 0.9732 | 0.0976 | Best Precision@5 overall. Weight 21 was almost tied at 0.5820. |
| Focused sweep 2 | `22..25` | 23 | 0.5812 | 2.906 | 0.6505 | 0.9726 | 0.0976 | Extension did not beat weight 20; performance starts to flatten/drop. |

The focused extension answered the main question: there is no need to keep pushing beyond `25`. Weight `20` remains the best Precision@5 setting across the tested range, while `21-23` are close but slightly lower. NDCG is marginally higher at some lower weights, but the differences are small and Precision@5 is the primary target for short recommendation lists.

Decision: use **`bayesian_uncertainty_weight=20`** as the current default. It gives the best observed Precision@5 on the current corpus and the 22-25 extension confirms that the high-weight curve has already peaked.
